## Step 4: Combining Multiple Runs Across Subjects
We have 3 subjects (S001, S002, S003) and 3 runs per subject (R03, R07, R11).
Each run only gives us ~8 left and ~7 right epochs per subject.
Combining all 9 runs across 3 subjects gives us enough data for reliable classification.

In [2]:
import mne
import numpy as np

In [9]:
# List of all runs
runs = [
    # Subject 1
    '../DATA/S001R03.edf',
    '../DATA/S001R07.edf', 
    '../DATA/S001R11.edf',
    # Subject 2
    '../DATA/S002R03.edf',
    '../DATA/S002R07.edf',
    '../DATA/S002R11.edf',
    # Subject 3
    '../DATA/S003R03.edf',
    '../DATA/S003R07.edf',
    '../DATA/S003R11.edf',
]
# Load, filter and extract epochs from each run
all_epochs = []
for run in runs:
    # Load
    raw = mne.io.read_raw_edf(run, preload=True, verbose=False)
    # Filter
    raw_filtered = raw.copy()
    raw_filtered.filter(8., 30., fir_design='firwin', verbose=False)
    # Get events
    events, _ = mne.events_from_annotations(raw, verbose=False)
    # Create epochs
    event_id_dict = {'rest': 1, 'left_fist': 2, 'right_fist': 3}
    epochs = mne.Epochs(raw_filtered, events, event_id=event_id_dict,
                        tmin=0.0, tmax=4.0, baseline=None,
                        preload=True, verbose=False)
    all_epochs.append(epochs)
    print(f"Loaded {run} — Left: {len(epochs['left_fist'])}, Right: {len(epochs['right_fist'])}")

Loaded ../DATA/S001R03.edf — Left: 8, Right: 7
Loaded ../DATA/S001R07.edf — Left: 8, Right: 7
Loaded ../DATA/S001R11.edf — Left: 7, Right: 8
Loaded ../DATA/S002R03.edf — Left: 8, Right: 7
Loaded ../DATA/S002R07.edf — Left: 7, Right: 8
Loaded ../DATA/S002R11.edf — Left: 8, Right: 7
Loaded ../DATA/S003R03.edf — Left: 7, Right: 8
Loaded ../DATA/S003R07.edf — Left: 8, Right: 7
Loaded ../DATA/S003R11.edf — Left: 7, Right: 8


In [10]:
# Combine all epochs together
combined_epochs = mne.concatenate_epochs(all_epochs)
print("Combined epochs:")
print("Total left fist:", len(combined_epochs['left_fist']))
print("Total right fist:", len(combined_epochs['right_fist']))
print("Total rest:", len(combined_epochs['rest']))

Not setting metadata
270 matching events found
No baseline correction applied
Combined epochs:
Total left fist: 68
Total right fist: 67
Total rest: 135


/var/folders/bx/lcstgrcx6jsg2vvgzpp2_wp00000gn/T/ipykernel_19904/1112968558.py:2: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  combined_epochs = mne.concatenate_epochs(all_epochs)


In [11]:
raw_filtered.save('../outputs/filtered_data/Combined_filtered_raw.fif', overwrite=True)
combined_epochs.save(
    '../outputs/filtered_data/S001_S002_S003_combined-epo.fif', 
    overwrite=True
)
print("Combined epochs saved!")

Writing /Users/kandulasatwika/Desktop/400_BCI/Notebooks/../outputs/filtered_data/Combined_filtered_raw.fif
Closing /Users/kandulasatwika/Desktop/400_BCI/Notebooks/../outputs/filtered_data/Combined_filtered_raw.fif
[done]
Overwriting existing file.
Overwriting existing file.
Combined epochs saved!
